# <font color="#418FDE" size="6.5" uppercase>**Distances Kernels**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Compute pairwise distances and similarities for small datasets. 
- Compare common distance metrics and kernel functions visually and numerically. 
- Use precomputed distance or kernel matrices with compatible estimators. 


## **1. Distance Metrics**

### **1.1. Pairwise Distance Matrix**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_01.jpg?v=1783989169" width="250">



>* Shows distances between every pair of items
>* Zero diagonal, symmetric, reveals hidden patterns

>* Compare every item using feature-based distances
>* Reveal relationships for clustering and similarity tasks

>* Prepare features to avoid distorted distances
>* Check matrix entries against expected similarity



In [ ]:
#@title Python Code - Pairwise Distance Matrix

# This example builds a small distance matrix.
# Distances compare every observation with every other.
# The heatmap reveals closest and farthest pairs.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

# Each row is one house with three numeric features.
house_names = ["A", "B", "C", "D"]
house_data = pd.DataFrame(
    {
        "area_sq_m": [90, 95, 160, 170],
        "bedrooms": [2, 2, 4, 4],
        "km_to_center": [3.0, 3.5, 12.0, 11.0],
    },
    index=house_names,
)

# Standardizing prevents area from dominating the distance.
scaler = StandardScaler()
scaled_values = scaler.fit_transform(house_data)

# Euclidean distance compares every scaled row to every scaled row.
distance_values = pairwise_distances(scaled_values, metric="euclidean")
distance_matrix = pd.DataFrame(
    np.round(distance_values, 2),
    index=house_names,
    columns=house_names,
)

# Validate the two key distance matrix properties.
diagonal_is_zero = np.allclose(np.diag(distance_values), 0.0)
matrix_is_symmetric = np.allclose(distance_values, distance_values.T)

print("Pairwise Euclidean distances after standardizing features:")
print(distance_matrix)
print("Diagonal all zero:", diagonal_is_zero)
print("Matrix symmetric:", matrix_is_symmetric)

# Mask the diagonal so a house is not closest to itself.
masked_distances = distance_values.copy()
np.fill_diagonal(masked_distances, np.inf)
closest_index = np.unravel_index(np.argmin(masked_distances), masked_distances.shape)

closest_pair = (house_names[closest_index[0]], house_names[closest_index[1]])
closest_distance = distance_values[closest_index]
print("Closest pair:", closest_pair, "distance", round(closest_distance, 2))

# A heatmap makes small and large distances easy to compare.
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(distance_values, cmap="Blues")

ax.set_xticks(range(len(house_names)))
ax.set_yticks(range(len(house_names)))
ax.set_xticklabels(house_names)
ax.set_yticklabels(house_names)

ax.set_xlabel("House")
ax.set_ylabel("House")
ax.set_title("Pairwise Distance Matrix")
fig.colorbar(image, ax=ax, label="Euclidean distance")

for row in range(len(house_names)):
    for col in range(len(house_names)):
        ax.text(col, row, f"{distance_values[row, col]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()



### **1.2. Euclidean Manhattan**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_02.jpg?v=1783989171" width="250">



>* Euclidean distance measures straight-line separation
>* Pairwise matrices store every point-to-point distance

>* Adds absolute feature differences across dimensions
>* Highlights additive changes in distance matrices

>* Match distance metric to data meaning
>* Scale features and compare distance matrices



In [ ]:
#@title Python Code - Euclidean Manhattan

# Compare two common distance metrics.
# Small points make pairwise distances visible.
# Matrices reveal different closeness patterns.

import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

# Each row is one observation with two numeric features.
points = np.array([[0, 0], [3, 4], [5, 1]], dtype=float)
labels = ["A", "B", "C"]

# Validate the tiny dataset before computing distances.
if points.shape != (3, 2):
    raise ValueError("Expected three points with two features.")

# Compute all pairwise Euclidean and Manhattan distances.
euclidean = pairwise_distances(points, metric="euclidean")
manhattan = pairwise_distances(points, metric="manhattan")

# Put the matrices in labeled tables for easier reading.
euclidean_table = pd.DataFrame(euclidean, index=labels, columns=labels)
manhattan_table = pd.DataFrame(manhattan, index=labels, columns=labels)

# Round values so beginners can compare the patterns quickly.
print("Points: A=(0,0), B=(3,4), C=(5,1)")
print("Euclidean distances:")
print(euclidean_table.round(2))
print("Manhattan distances:")
print(manhattan_table.round(2))



### **1.3. Custom Distance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_03.jpg?v=1783989173" width="250">



>* Define similarity using domain knowledge
>* Inspect small distance matrices by hand

>* Choose and weight features by task importance
>* Define comparisons for mixed data types

>* Test custom distances with small examples
>* Ensure clear, valid, context-matching metrics



In [ ]:
#@title Python Code - Custom Distance Metrics

# This example builds a custom distance metric.
# Feature weights encode what similarity should mean.
# The output compares standard and custom distances.

import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

# Each row describes one apartment using simple numeric features.
apartments = pd.DataFrame(
    {
        "rent_dollars": [1200, 1250, 1800, 1220],
        "commute_minutes": [25, 28, 20, 55],
        "bedrooms": [1, 1, 2, 1],
        "floor": [2, 10, 3, 2],
    },
    index=["A", "B", "C", "D"],
)

# Scaling keeps large dollar values from dominating every comparison.
scaled_values = apartments.copy()
scaled_values["rent_dollars"] = scaled_values["rent_dollars"] / 100
scaled_values["commute_minutes"] = scaled_values["commute_minutes"] / 10

# These weights say rent and commute matter more than floor number.
feature_weights = np.array([2.0, 1.5, 3.0, 0.2])

# This function computes a weighted Manhattan distance between two rows.
def weighted_apartment_distance(row_a, row_b):
    absolute_differences = np.abs(row_a - row_b)
    weighted_differences = absolute_differences * feature_weights
    return np.sum(weighted_differences)

# Pairwise distances compare every apartment with every other apartment.
standard_distances = pairwise_distances(scaled_values, metric="manhattan")
custom_distances = pairwise_distances(
    scaled_values,
    metric=weighted_apartment_distance,
)

# A small validation confirms the distance matrix has the expected shape.
expected_shape = (len(apartments), len(apartments))
if custom_distances.shape != expected_shape:
    raise ValueError("The distance matrix should be square.")

# Display only the distances from apartment A for easy comparison.
standard_from_a = pd.Series(standard_distances[0], index=apartments.index)
custom_from_a = pd.Series(custom_distances[0], index=apartments.index)
comparison = pd.DataFrame(
    {"standard": standard_from_a, "custom": custom_from_a}
).round(2)

print("Distances from apartment A to each apartment:")
print(comparison.to_string())
print("Closest to A using the custom metric:", custom_from_a.iloc[1:].idxmin())



## **2. Similarity Kernels**

### **2.1. Cosine Similarity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_01.jpg?v=1783989177" width="250">



>* Compares vector direction, not overall size
>* Angles show similar, unrelated, or opposite patterns

>* Higher cosine means more similar patterns
>* Cosine compares direction, not absolute distance

>* Build matrices showing shared directional patterns
>* Use carefully when magnitude matters



In [ ]:
#@title Python Code - Cosine Similarity

# This example compares cosine similarity values.
# Direction matters more than vector length.
# The heatmap reveals pattern-based similarity.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

# Rows represent simple purchase patterns across categories.
customer_names = ["A small books", "B large books", "C electronics", "D mixed"]
purchases = np.array([[2, 1, 0], [20, 10, 0], [0, 1, 3], [3, 2, 1]])

# Validate that cosine similarity can be computed safely.
row_lengths = np.linalg.norm(purchases, axis=1)
if np.any(row_lengths == 0):
    raise ValueError("Every row needs at least one nonzero feature.")

# Cosine similarity compares directions, not total purchase volume.
similarity = cosine_similarity(purchases)
similarity_table = pd.DataFrame(similarity, customer_names, customer_names)

# Euclidean distance shows that magnitude can tell a different story.
euclidean_ab = np.linalg.norm(purchases[0] - purchases[1])
cosine_ab = similarity[0, 1]

print("Cosine similarity between A and B:", round(cosine_ab, 3))
print("Euclidean distance between A and B:", round(euclidean_ab, 3))
print("A and B have the same proportions, but different totals.")

# A heatmap makes the full similarity kernel easy to compare.
plt.figure(figsize=(7, 5))
ax = sns.heatmap(
    similarity_table,
    annot=True,
    cmap="viridis",
    vmin=0,
    vmax=1,
    fmt=".2f",
)

ax.set_title("Cosine Similarity Kernel for Purchase Patterns")
ax.set_xlabel("Customer")
ax.set_ylabel("Customer")
plt.tight_layout()
plt.show()



### **2.2. Linear Polynomial Kernel**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_02.jpg?v=1783989175" width="250">



>* Measures feature alignment and shared magnitudes
>* Heat maps reveal similar observation groups

>* Captures feature interactions for richer similarity
>* Creates sharper clusters in kernel heat maps

>* Scale features before comparing kernel values
>* Choose kernels using performance and visual patterns



In [ ]:
#@title Python Code - Linear Polynomial Kernel

# Compare linear and polynomial kernel similarities.
# Scaling keeps feature sizes from dominating.
# Heatmap shows stronger polynomial contrast.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import polynomial_kernel
from sklearn.preprocessing import StandardScaler

# Small rows represent customers and product-category purchases.
customer_names = ["A", "B", "C", "D", "E"]
purchases = np.array([[8, 7], [7, 8], [2, 3], [3, 2], [9, 1]], dtype=float)

# Validate the tiny matrix before computing pairwise similarities.
if purchases.shape != (5, 2):
    raise ValueError("Expected five customers with two numeric features.")

# Standardization makes both product categories equally influential.
scaler = StandardScaler()
scaled_purchases = scaler.fit_transform(purchases)

# Linear similarity uses direct alignment of scaled feature values.
linear_scores = linear_kernel(scaled_purchases)

# Polynomial similarity also emphasizes feature interactions.
poly_scores = polynomial_kernel(
    scaled_purchases,
    degree=2,
    gamma=1,
    coef0=1,
)

# Compare one pair that looks similar and one pair that differs.
print("Pair A-B linear:", round(linear_scores[0, 1], 2))
print("Pair A-B polynomial:", round(poly_scores[0, 1], 2))
print("Pair A-E linear:", round(linear_scores[0, 4], 2))
print("Pair A-E polynomial:", round(poly_scores[0, 4], 2))

# Plot the polynomial matrix to reveal amplified similarity blocks.
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(poly_scores, cmap="viridis")

ax.set_title("Polynomial Kernel Similarity")
ax.set_xlabel("Customer")
ax.set_ylabel("Customer")

ax.set_xticks(np.arange(len(customer_names)))
ax.set_yticks(np.arange(len(customer_names)))
ax.set_xticklabels(customer_names)
ax.set_yticklabels(customer_names)

fig.colorbar(image, ax=ax, label="Similarity score")
plt.tight_layout()
plt.show()



### **2.3. RBF Laplacian Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_03.jpg?v=1783989178" width="250">



>* Convert distances into similarity scores
>* Nearby points influence each other most

>* RBF fades smoothly; Laplacian drops more sharply
>* Kernel shape changes neighborhood grouping behavior

>* Scale controls how quickly similarity fades
>* Matrices and heatmaps reveal useful structure



In [ ]:
#@title Python Code - RBF Laplacian Kernels

# Compare RBF and Laplacian similarity kernels.
# Distances become similarities with different decay shapes.
# The plot shows how scale changes similarity.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics.pairwise import laplacian_kernel

# Distances are one-dimensional points from a reference point.
distances = np.linspace(0.0, 4.0, 81)
points = distances.reshape(-1, 1)
reference = np.array([[0.0]])

# Gamma controls how quickly similarity fades with distance.
gamma_small = 0.5
gamma_large = 2.0

# RBF uses squared Euclidean distance inside the exponential.
rbf_small = rbf_kernel(points, reference, gamma=gamma_small).ravel()
rbf_large = rbf_kernel(points, reference, gamma=gamma_large).ravel()

# Laplacian uses absolute distance inside the exponential.
lap_small = laplacian_kernel(points, reference, gamma=gamma_small).ravel()
lap_large = laplacian_kernel(points, reference, gamma=gamma_large).ravel()

# Validate that every curve matches the distance grid.
if rbf_small.shape != distances.shape:
    raise ValueError("Kernel output should match the distance grid.")

print("Similarity at distance 0 is 1.0 for both kernels.")
print("At distance 2, RBF gamma 0.5:", round(float(rbf_small[40]), 3))
print("At distance 2, Laplacian gamma 0.5:", round(float(lap_small[40]), 3))
print("At distance 2, RBF gamma 2.0:", round(float(rbf_large[40]), 3))
print("At distance 2, Laplacian gamma 2.0:", round(float(lap_large[40]), 3))

# Plot all curves on one axes for direct visual comparison.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(distances, rbf_small, label="RBF, gamma=0.5")
ax.plot(distances, lap_small, label="Laplacian, gamma=0.5")
ax.plot(distances, rbf_large, label="RBF, gamma=2.0")
ax.plot(distances, lap_large, label="Laplacian, gamma=2.0")

# Labels make the distance-to-similarity transformation explicit.
ax.set_title("RBF and Laplacian Kernel Similarity Decay")
ax.set_xlabel("Distance from reference point")
ax.set_ylabel("Similarity score")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()



## **3. Special Kernels**

### **3.1. Chi Squared Kernel**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_01.jpg?v=1783989181" width="250">



>* Best for nonnegative histogram-like data
>* Compares relative distribution shapes, not magnitudes

>* Estimators use pairwise chi squared similarities
>* Histogram kernels capture domain-specific distribution patterns

>* Use only comparable nonnegative histogram features
>* Normalize and match kernel to data meaning



In [ ]:
#@title Python Code - Chi Squared Kernel

# This example builds a chi squared kernel matrix.
# Histogram samples must be nonnegative and comparable.
# A precomputed SVM uses the similarity matrix.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import chi2_kernel
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import sklearn

# Each row is a small normalized histogram.
X = np.array([
    [0.70, 0.20, 0.10], [0.65, 0.25, 0.10], [0.60, 0.30, 0.10],
    [0.10, 0.25, 0.65], [0.15, 0.20, 0.65], [0.10, 0.30, 0.60],
    [0.55, 0.35, 0.10], [0.20, 0.20, 0.60], [0.75, 0.15, 0.10],
    [0.15, 0.30, 0.55], [0.58, 0.32, 0.10], [0.12, 0.28, 0.60],
])

# Labels separate red-heavy and blue-heavy histograms.
y = np.array([0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1])

# Validate the key chi squared kernel assumption.
if np.any(X < 0):
    raise ValueError("Chi squared kernels require nonnegative features.")

# Split first, then build train and test kernel matrices.
indices = np.arange(X.shape[0])
train_idx, test_idx = train_test_split(
    indices, test_size=0.33, random_state=42, stratify=y
)

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

# gamma controls how quickly similarity decreases.
gamma = 2.0
K_train = chi2_kernel(X_train, X_train, gamma=gamma)
K_test = chi2_kernel(X_test, X_train, gamma=gamma)

# The SVM receives similarities, not original features.
model = SVC(kernel="precomputed")
model.fit(K_train, y_train)

predicted = model.predict(K_test)
accuracy = accuracy_score(y_test, predicted)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Train kernel shape: {K_train.shape}")
print(f"Test accuracy with precomputed chi squared kernel: {accuracy:.2f}")

# Show the pairwise similarities used for training.
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(K_train, cmap="viridis", vmin=0, vmax=1)

ax.set_title("Precomputed chi squared kernel")
ax.set_xlabel("Training sample index")
ax.set_ylabel("Training sample index")
fig.colorbar(image, ax=ax, label="Similarity")

plt.show()



### **3.2. Precomputed Matrices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_02.jpg?v=1783989183" width="250">



>* Precompute pairwise distances or similarities externally
>* Connect domain measures to compatible estimators

>* Match matrix type to estimator expectations
>* Keep rows, columns, and labels aligned

>* Compute new-to-training matrices consistently
>* Keep folds ordered and dimensions compatible



In [ ]:
#@title Python Code - Precomputed Matrices

# Demonstrate precomputed matrices with a small classifier.
# Distances become similarities before model training.
# The output compares correct and shuffled alignment.

import numpy as np
import sklearn
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Create a tiny two-class dataset for clear pairwise relationships.
features, labels = make_blobs(
    n_samples=40,
    centers=2,
    cluster_std=1.2,
    random_state=42,
)

# Split first, because test kernels compare test rows to training rows.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Convert distances into an RBF similarity kernel matrix.
gamma = 0.2
train_distances = pairwise_distances(train_features, train_features)
train_kernel = np.exp(-gamma * train_distances ** 2)

# Validate the square training matrix expected by precomputed kernels.
if train_kernel.shape != (len(train_labels), len(train_labels)):
    raise ValueError("Training kernel must be square and label-aligned.")

# Fit one SVM that receives similarities instead of raw features.
model = SVC(kernel="precomputed")
model.fit(train_kernel, train_labels)

# Prediction needs test-to-training similarities, not a square test matrix.
test_distances = pairwise_distances(test_features, train_features)
test_kernel = np.exp(-gamma * test_distances ** 2)
predictions = model.predict(test_kernel)

# Shuffle labels only to show why matrix-label order matters.
shuffled_train_labels = train_labels[::-1]
misaligned_model = SVC(kernel="precomputed")
misaligned_model.fit(train_kernel, shuffled_train_labels)
misaligned_predictions = misaligned_model.predict(test_kernel)

# Print concise results that highlight dimensions and alignment.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Training kernel shape: {train_kernel.shape}")
print(f"Test kernel shape: {test_kernel.shape}")
print(f"Accuracy with aligned labels: {accuracy_score(test_labels, predictions):.2f}")
print(f"Accuracy with shuffled labels: {accuracy_score(test_labels, misaligned_predictions):.2f}")



### **3.3. Kernel Limitations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_03.jpg?v=1783989185" width="250">



>* Kernel matrices must represent valid similarities
>* Invalid kernels can cause unreliable model results

>* Kernel matrices grow quickly with dataset size
>* Updates require many new similarity calculations

>* Kernel choice shapes learnable patterns
>* Validate kernels and expect harder interpretation



# <font color="#418FDE" size="6.5" uppercase>**Distances Kernels**</font>


In this lecture, you learned to:
- Compute pairwise distances and similarities for small datasets. 
- Compare common distance metrics and kernel functions visually and numerically. 
- Use precomputed distance or kernel matrices with compatible estimators. 

In the next Lecture (Lecture B), we will go over 'Kernel Approximation'